In [1]:
import sys
# Bu komut, kernelin kullandığı python yolunu (executable) bulur 
# ve indirmeyi doğrudan oraya yapar.
!{sys.executable} -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 10.7 MB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 703.3/703.3 kB 1.8 MB/s eta 0:00:00-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [en-core-web-trf] [en-core-web-trf]

[notice] A new release of pip is available: 25.1.1 -> 26.0
[notice] To update, run: pip3.11 install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


In [2]:
import pandas as pd
import spacy
from transformers import pipeline

# 1. Veriyi Yükle
input_file = "../data/labels/auto_transcripts.csv"  # Dosya adınızın doğru olduğundan emin olun
try:
    df = pd.read_csv(input_file)
    print(f"{len(df)} satır veri yüklendi.")
except FileNotFoundError:
    print("Hata: CSV dosyası bulunamadı.")
    exit()

# --- MODELLERİN YÜKLENMESİ ---
print("Yapay zeka modelleri yükleniyor (ilk seferde indirme yapabilir)...")

# A) Konum Tespiti (NER) - Adresleri bulmak için
# Apple Silicon (M1/M2/M3) kullanıyorsanız GPU hızlandırması için spacy.prefer_gpu() ekleyebilirsiniz.
nlp_ner = spacy.load("en_core_web_trf")

# B) Olay Sınıflandırma (Zero-Shot) - Olayın ne olduğunu anlamak için
# 'bart-large-mnli' modeli etiketlenmemiş verilerde harika çalışır.
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# C) Duygu Analizi (Sentiment) - Panik durumunu ölçmek için
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# --- ANALİZ FONKSİYONLARI ---

def extract_location(text):
    """Metin içindeki lokasyon, cadde ve şehir isimlerini çeker."""
    if not isinstance(text, str): return ""
    doc = nlp_ner(text)
    # GPE: Ülke/Şehir, LOC: Bölge/Su, FAC: Bina/Havalimanı vb.
    locations = [ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC", "FAC"]]
    return ", ".join(list(set(locations))) if locations else "Tespit Edilemedi"

def analyze_event(text):
    """Metni analiz edip en olası olay türünü ve birimi belirler."""
    if not isinstance(text, str): return "Bilinmiyor", "Bilinmiyor"
    
    # Modele vereceğimiz olası etiketler
    candidate_labels = ["medical emergency", "fire", "shooting", "robbery", "traffic accident", "domestic violence", "heart attack", "overdose"]
    
    # Sınıflandırma yap
    result = classifier(text, candidate_labels)
    event_type = result['labels'][0]  # En yüksek skorlu tahmin
    
    # Olay türüne göre birim atama (Basit mantık)
    if event_type in ["medical emergency", "heart attack", "overdose"]:
        unit = "Ambulance / EMS"
    elif event_type in ["fire"]:
        unit = "Fire Department"
    else:
        unit = "Police"
        
    return event_type, unit

def analyze_sentiment(text):
    """Metnin duygu durumunu (NEGATIVE=Panik/Korku, POSITIVE=Sakin) ölçer."""
    if not isinstance(text, str): return "Nötr"
    # Metin çok uzunsa kırp (Model limiti genelde 512 token)
    result = sentiment_analyzer(text[:512])[0]
    label = result['label']
    score = result['score']
    
    if label == "NEGATIVE" and score > 0.9:
        return "Yüksek Panik/Acil"
    elif label == "NEGATIVE":
        return "Endişeli"
    else:
        return "Sakin/Bilgilendirici"

# --- İŞLEME DÖNGÜSÜ ---
print("Analiz başlıyor, bu işlem bilgisayar hızınıza göre 10-20 dk sürebilir...")

results = []

for index, row in df.iterrows():
    text = row['text']
    
    # İlerlemeyi göster
    if index % 10 == 0:
        print(f"İşleniyor: {index}/{len(df)}")
    
    loc = extract_location(text)
    event, unit = analyze_event(text)
    sentiment = analyze_sentiment(text)
    
    results.append({
        "Dosya_Yolu": row.get('path', ''),
        "Orijinal_Metin": text,
        "Tahmin_Edilen_Olay": event,
        "Yonlendirilen_Birim": unit,
        "Tespit_Edilen_Konum": loc,
        "Duygu_Durumu": sentiment
    })

# Sonuçları Kaydet
df_results = pd.DataFrame(results)
df_results.to_csv("Outputs/Text Analysis Outputs/911_ai_sonuc.csv", index=False)
print("İşlem tamamlandı! '911_ai_sonuc.csv' dosyası oluşturuldu.")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


706 satır veri yüklendi.
Yapay zeka modelleri yükleniyor (ilk seferde indirme yapabilir)...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2139.16it/s, Materializing param=pre_classifier.weight]                                  


Analiz başlıyor, bu işlem bilgisayar hızınıza göre 10-20 dk sürebilir...
İşleniyor: 0/706
İşleniyor: 10/706
İşleniyor: 20/706
İşleniyor: 30/706
İşleniyor: 40/706
İşleniyor: 50/706
İşleniyor: 60/706
İşleniyor: 70/706
İşleniyor: 80/706
İşleniyor: 90/706
İşleniyor: 100/706
İşleniyor: 110/706
İşleniyor: 120/706
İşleniyor: 130/706
İşleniyor: 140/706
İşleniyor: 150/706
İşleniyor: 160/706
İşleniyor: 170/706
İşleniyor: 180/706
İşleniyor: 190/706
İşleniyor: 200/706
İşleniyor: 210/706
İşleniyor: 220/706
İşleniyor: 230/706
İşleniyor: 240/706
İşleniyor: 250/706
İşleniyor: 260/706
İşleniyor: 270/706
İşleniyor: 280/706
İşleniyor: 290/706
İşleniyor: 300/706
İşleniyor: 310/706
İşleniyor: 320/706
İşleniyor: 330/706
İşleniyor: 340/706
İşleniyor: 350/706
İşleniyor: 360/706
İşleniyor: 370/706
İşleniyor: 380/706
İşleniyor: 390/706
İşleniyor: 400/706
İşleniyor: 410/706
İşleniyor: 420/706
İşleniyor: 430/706
İşleniyor: 440/706
İşleniyor: 450/706
İşleniyor: 460/706
İşleniyor: 470/706
İşleniyor: 480/706
İşleniy